In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix

from src.utils import get_pytorch_loader
from src.train import CNNModel
from src.config import DEVICE, PYTORCH_MODEL_PATH

loader = get_pytorch_loader()

model = CNNModel().to(DEVICE)

model.load_state_dict(
    torch.load(
        PYTORCH_MODEL_PATH,
        map_location=DEVICE
    )
)

model.eval()

y_true = []
y_pred = []
misclassified_images = []

with torch.no_grad():
    for images, labels in loader:
        images = images.to(DEVICE)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        y_true.extend(labels.numpy())
        y_pred.extend(predictions.cpu().numpy())

        for i in range(len(labels)):
            if predictions[i].cpu().item() != labels[i].item():
                misclassified_images.append(
                    (
                        images[i].cpu(),
                        labels[i].item(),
                        predictions[i].cpu().item()
                    )
                )

y_true = np.array(y_true)
y_pred = np.array(y_pred)


accuracy = (y_true == y_pred).mean()

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Wrong Predictions: {(y_true != y_pred).sum()}")

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()


for i in range(min(5, len(misclassified_images))):
    image, true_label, pred_label = misclassified_images[i]

    image = image.permute(1, 2, 0).numpy()
    image = (image * 0.5) + 0.5

    plt.figure(figsize=(4, 4))
    plt.imshow(image)

    plt.title(
        f"True: {true_label} | Predicted: {pred_label}"
    )

    plt.axis("off")
    plt.show()

ModuleNotFoundError: No module named 'src.dataset'